# Save tagged-unit figures

Independent notebook that loads one opto-tagging session, selects the **tagged units**
(significant, QC-passed, for a chosen condition/probe/shank), and saves a **separate**
raster + PSTH figure for **each** unit. Each figure includes the unit's mean waveform
as an inset in the top-right corner (`show_waveform=True`).

This notebook only reads the previously computed significance CSV, so it does not
recompute `find_significant_units`.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

In [ ]:
# =============================================================================
# 2. LOAD SESSION
# =============================================================================
import glob
from optical_tagging import OpticalTagging

session_date = '863973_2026-08-25_14-18-29'
behavior_json_file = f'/root/capsule/data/ecephys_{session_date}/behavior/{session_date}.json'

# The sorted folder has an extra timestamp not contained in session_date,
# so locate it by glob (one match expected per session).
ephys_nwb_file = glob.glob(
    f'/root/capsule/data/ecephys_{session_date}_sorted_*/nwb/ecephys_{session_date}_experiment1_recording1.nwb'
)[0]

OptoTag = OpticalTagging(
    behavior_json_file=behavior_json_file,
    ephys_nwb_file=ephys_nwb_file,
)

sig_csv_path = f'/root/capsule/scratch/opto_tagging/sig_{session_date}.csv'
print(f"OptoTag ready. Significance CSV: {sig_csv_path}")

In [ ]:
# =============================================================================
# 3. SELECT TAGGED UNITS (significant + QC-passed) FOR ONE CONDITION
# =============================================================================
import types
import pandas as pd

# ---- Choose the condition / probe / shank to plot --------------------------
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)
probe = "ProbeC-2"
shank = 1

# Load the significance CSV and parse the 'condition' column, which is stored
# as strings like "(np.float64(5.0), ...)".
df = pd.read_csv(sig_csv_path)
fake_np = types.SimpleNamespace(float64=float)

def safe_parse_condition(x):
    if isinstance(x, str):
        try:
            return eval(x, {"np": fake_np})
        except Exception as e:
            print(f"Failed to parse: {x}\nError: {e}")
            return None
    return x

df['condition'] = df['condition'].apply(safe_parse_condition)

# Significant, positively-modulated units for this condition/probe/shank.
sigrows = df[
    (df["condition"] == target_cond) &
    (df["significant"] == True) &
    (df["stim_FR_mean"] > df["baseline_FR_mean"]) &
    (df["probe"] == probe) &
    (df["shank"] == shank)
]

# Intersect with QC-passed units on the OptoTag object.
common_units = sorted(set(OptoTag.units_passing_qc.index) & set(sigrows['unit_id']))
print(f"Condition: {target_cond} | {probe} shank {shank}")
print(f"Tagged units to plot ({len(common_units)}): {common_units}")

In [ ]:
# =============================================================================
# 4. SAVE ONE FIGURE PER TAGGED UNIT (raster + PSTH + waveform inset)
# =============================================================================
import os

# Output folder for the per-unit figures.
tagged_fig_dir = f"/root/capsule/results/tagged_units/{session_date}/{probe}_shank{shank}/"
os.makedirs(tagged_fig_dir, exist_ok=True)

# plot_raster_graph supports 'pdf' and 'eps' (add 'png' if your version handles it).
SAVE_FORMATS = ["png", "eps"]

print(f"Saving {len(common_units)} tagged-unit figures to: {tagged_fig_dir}")
for u in common_units:
    # Per-unit path prefix -> plot_raster_graph appends the condition + extension,
    # so every unit (and condition) lands in its own file.
    unit_save_path = os.path.join(tagged_fig_dir, f"unit_{u}")
    OptoTag.plot_raster_graph(
        unit_index=u,
        bin_size=0.002,
        time_window=[-0.02, 0.09],
        removal_window=0.002,
        align_to_event="pulse",
        remove_artefacts=True,
        save_path=unit_save_path,
        save_formats=SAVE_FORMATS,
        show_waveform=True,   # mean waveform in the top-right corner
    )
    print(f"  saved unit {u}")

print("Done saving tagged-unit figures.")